In [5]:
import uuid
import time
import random
from typing import TypedDict, Dict, Any, Optional

# LangGraph 相关
from langgraph.graph import START, END, StateGraph
from langchain_core.runnables import RunnableConfig

# ==========================================
# 1. 系统配置
# ==========================================
MAX_RETRIES = 3
DEFAULT_TASK_BUDGET = 5.0  # 默认全局 Deadline 预算 (秒)
DEFAULT_TIMEOUT = 2.0      # 单次请求 Request Timeout (秒)

# ==========================================
# 2. 定义状态 (State)
# ==========================================
class AgentState(TypedDict):
    employee_id: str
    amount: int
    idempotency_key: str
    retry_count: int
    status: str
    error_code: int
    error_msg: str
    result: dict

    # Day 22 控制参数
    start_time: float          # 任务起始时间戳
    deadline: float            # 绝对截止时间戳 (start_time + task_budget)
    remaining_budget: float    # 剩余预算 (秒)
    request_timeout: float     # 单次请求超时
    cancelled: bool            # 取消标记

    # 测试用参数
    test_scenario: str         # 指定测试场景名称
    mock_attempt: int          # Mock API 内部计数

# ==========================================
# 3. 核心算法: Exponential Backoff + Jitter
# ==========================================
def calculate_retry_delay(
    retry_count: int,
    base_delay: float = 0.5,
    max_delay: float = 3.0,
    random_fn=random.uniform,
) -> float:
    max_backoff = min(base_delay * (2 ** retry_count), max_delay)
    return random_fn(0, max_backoff)

def update_remaining_budget(state: AgentState) -> float:
    """计算实时剩余预算"""
    now = time.time()
    remaining = max(0.0, state["deadline"] - now)
    return remaining

def log_policy_status(
    attempt: int,
    error_code: int,
    retry_count: int,
    remaining_budget: float,
    cancelled: bool,
    backoff: Optional[float] = None
):
    """可视化 Policy 日志格式化打印"""
    print(f"\n[API] Attempt #{attempt}")
    print(f"[API] {error_code}")
    print(f"\n[Policy]")
    print(f"retry_count = {retry_count}/{MAX_RETRIES}")
    print(f"remaining_budget = {remaining_budget:.2f}s")
    print(f"cancelled = {cancelled}")
    if backoff is not None:
        print(f"\n[Retry]")
        print(f"backoff = {backoff:.2f}s")

# ==========================================
# 4. 图节点定义 (Nodes)
# ==========================================

def prepare_operation_node(state: AgentState) -> Dict[str, Any]:
    now = time.time()
    task_budget = state.get("remaining_budget", DEFAULT_TASK_BUDGET)
    deadline = now + task_budget
    operation_id = state.get("idempotency_key") or str(uuid.uuid4())
    
    print(f"\n==========================================")
    print(f"🚀 [1. 准备阶段] 初始化任务")
    print(f"   -> 幂等键: {operation_id}")
    print(f"   -> 全局 Deadline 预算: {task_budget:.2f}s")
    print(f"==========================================")

    return {
        "idempotency_key": operation_id,
        "retry_count": 0,
        "start_time": now,
        "deadline": deadline,
        "remaining_budget": task_budget,
        "request_timeout": state.get("request_timeout", DEFAULT_TIMEOUT),
        "cancelled": state.get("cancelled", False),
        "mock_attempt": 0,
        "status": "prepared"
    }

def call_hr_api_node(state: AgentState) -> Dict[str, Any]:
    """模拟 HR API 调用，根据场景逻辑控制返回"""
    attempt = state.get("retry_count", 0) + 1
    mock_attempt = state.get("mock_attempt", 0) + 1
    scenario = state.get("test_scenario", "SUCCESS")
    
    # 场景 1: 始终成功
    if scenario == "SUCCESS":
        status_code = 200
    # 场景 2: 第一次 503，第二次 200
    elif scenario == "RETRY_SUCCESS":
        status_code = 503 if mock_attempt == 1 else 200
    # 场景 3 & 4: 抛出 503 触发错误判断
    elif scenario in ["DEADLINE_EXHAUSTED", "CANCELLED"]:
        status_code = 503
    else:
        status_code = 503

    rem_budget = update_remaining_budget(state)
    
    if status_code == 200:
        print(f"\n[API] Attempt #{attempt}")
        print(f"[API] 200 OK")
        return {
            "status": "SUCCESS",
            "result": {"success": True, "message": "HR Bonus applied."},
            "remaining_budget": rem_budget,
            "mock_attempt": mock_attempt
        }
    else:
        return {
            "status": "api_error",
            "error_code": status_code,
            "error_msg": f"HTTP {status_code}",
            "remaining_budget": rem_budget,
            "mock_attempt": mock_attempt
        }

def handle_policy_node(state: AgentState, config: RunnableConfig) -> Dict[str, Any]:
    """核心 Policy 节点：依次评估 Cancellation -> Deadline Budget -> Retry Limit"""
    sleep_fn = config.get("configurable", {}).get("sleep_fn", time.sleep)
    
    attempt = state.get("retry_count", 0) + 1
    current_retry = state.get("retry_count", 0)
    error_code = state.get("error_code", 503)
    cancelled = state.get("cancelled", False)
    rem_budget = update_remaining_budget(state)

    # 1. 检查 Cancellation
    if cancelled:
        log_policy_status(attempt, error_code, current_retry, rem_budget, cancelled)
        print(f"\n[Policy Decision] 🛑 收到取消信号，不再 Retry。")
        return {"status": "CANCELLED", "remaining_budget": rem_budget}

    # 计算预计所需的重试 Backoff 延迟
    backoff = calculate_retry_delay(current_retry)

    # 2. 检查 Deadline Budget 是否支持本次重试
    if rem_budget <= 0 or rem_budget < backoff:
        log_policy_status(attempt, error_code, current_retry, rem_budget, cancelled)
        print(f"\n[Policy Decision] ⏰ Retry Budget 虽有，但 Deadline Budget 不足 ({rem_budget:.2f}s < backoff {backoff:.2f}s)。放弃调用。")
        return {"status": "DEADLINE_EXHAUSTED", "remaining_budget": rem_budget}

    # 3. 检查 Retry Limit
    if current_retry >= MAX_RETRIES:
        log_policy_status(attempt, error_code, current_retry, rem_budget, cancelled)
        print(f"\n[Policy Decision] ⚠️ 重试次数已达到上限 ({MAX_RETRIES})，转入 Fallback。")
        return {"status": "RETRY_EXHAUSTED", "remaining_budget": rem_budget}

    # 4. 允许重试，进行 Backoff 并扣减预算
    log_policy_status(attempt, error_code, current_retry, rem_budget, cancelled, backoff=backoff)
    print(f"\n[Retry] 执行休眠 {backoff:.2f}s...")
    
    sleep_fn(backoff)
    
    post_sleep_budget = update_remaining_budget(state)
    return {
        "status": "retry_ready",
        "retry_count": current_retry + 1,
        "remaining_budget": post_sleep_budget
    }

# --- 终态 Node 定义 ---
def success_node(state: AgentState) -> Dict[str, Any]:
    print("\n✅ [Final State] SUCCESS - 操作执行成功")
    return {"result": {"status": "SUCCESS", "detail": state.get("result")}}

def deadline_exceeded_node(state: AgentState) -> Dict[str, Any]:
    print("\n⏰ [Final State] DEADLINE_EXCEEDED - 任务响应超时/预算耗尽")
    return {"result": {"status": "DEADLINE_EXCEEDED", "reason": "Task execution budget limit reached"}}

def cancelled_node(state: AgentState) -> Dict[str, Any]:
    print("\n🛑 [Final State] CANCELLED - 操作已被用户或外部信号取消")
    return {"result": {"status": "CANCELLED", "reason": "Request cancelled by caller"}}

def fallback_node(state: AgentState) -> Dict[str, Any]:
    print("\n⚠️ [Final State] FALLBACK - 重试耗尽，已推入异步补偿队列")
    return {"result": {"status": "FALLBACK", "reason": "Retry budget exhausted"}}

# ==========================================
# 5. 路由逻辑 (Router Functions)
# ==========================================
def route_after_api(state: AgentState) -> str:
    if state["status"] == "SUCCESS":
        return "success"
    return "handle_policy"

def route_after_policy(state: AgentState) -> str:
    status = state.get("status")
    status_map = {
        "retry_ready": "retry",
        "DEADLINE_EXHAUSTED": "deadline_exceeded",
        "CANCELLED": "cancelled",
        "RETRY_EXHAUSTED": "fallback"
    }
    
    if status not in status_map:
        raise KeyError(f"未知的 Policy 状态: '{status}'，无法进行路由！")
        
    return status_map[status]

# ==========================================
# 6. 构建 StateGraph
# ==========================================
builder = StateGraph(AgentState)

# 注册节点
builder.add_node("prepare", prepare_operation_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("handle_policy", handle_policy_node)

builder.add_node("success_node", success_node)
builder.add_node("deadline_exceeded_node", deadline_exceeded_node)
builder.add_node("cancelled_node", cancelled_node)
builder.add_node("fallback_node", fallback_node)

# 注册边
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "call_hr_api")

# 使用 explicit dict 映射规避 LangGraph 路由 key 冲突
builder.add_conditional_edges(
    "call_hr_api",
    route_after_api,
    {
        "success": "success_node",
        "handle_policy": "handle_policy"
    }
)

builder.add_conditional_edges(
    "handle_policy",
    route_after_policy,
    {
        "retry": "call_hr_api",
        "deadline_exceeded": "deadline_exceeded_node",
        "cancelled": "cancelled_node",
        "fallback": "fallback_node"
    }
)

builder.add_edge("success_node", END)
builder.add_edge("deadline_exceeded_node", END)
builder.add_edge("cancelled_node", END)
builder.add_edge("fallback_node", END)

graph = builder.compile()

# ==========================================
# 7. 测试运行 4 个 Scenario
# ==========================================
if __name__ == "__main__":
    
    def mock_sleep(seconds: float):
        time.sleep(seconds)

    run_config = {"configurable": {"sleep_fn": mock_sleep}}

    test_scenarios = [
        {
            "name": "Scenario 1 — Success (HR API 正常成功)",
            "scenario": "SUCCESS",
            "budget": 5.0,
            "cancelled": False
        },
        {
            "name": "Scenario 2 — Retry Success (第一次 503 -> Backoff -> Retry 成功)",
            "scenario": "RETRY_SUCCESS",
            "budget": 5.0,
            "cancelled": False
        },
        {
            "name": "Scenario 3 — Deadline Exhausted (第一次 503 -> Retry Budget 够但 Deadline 不足)",
            "scenario": "DEADLINE_EXHAUSTED",
            "budget": 0.2,  # 设置预算 < backoff，触发 DEADLINE_EXCEEDED
            "cancelled": False
        },
        {
            "name": "Scenario 4 — Cancelled (第一次 503 -> cancelled=True -> 不 Retry 直接取消)",
            "scenario": "CANCELLED",
            "budget": 5.0,
            "cancelled": True
        }
    ]

    for item in test_scenarios:
        print(f"\n\n##################################################")
        print(f"  {item['name']}")
        print(f"##################################################")
        
        final_state = graph.invoke(
            {
                "employee_id": "EMP_888",
                "amount": 1000,
                "test_scenario": item["scenario"],
                "remaining_budget": item["budget"],
                "cancelled": item["cancelled"]
            },
            config=run_config
        )
        print(f"\n[Execution Result]: {final_state['result']}")



##################################################
  Scenario 1 — Success (HR API 正常成功)
##################################################

🚀 [1. 准备阶段] 初始化任务
   -> 幂等键: 93df9e96-1cea-4bf8-84ec-0f5926f473ca
   -> 全局 Deadline 预算: 5.00s

[API] Attempt #1
[API] 200 OK

✅ [Final State] SUCCESS - 操作执行成功

[Execution Result]: {'status': 'SUCCESS', 'detail': {'success': True, 'message': 'HR Bonus applied.'}}


##################################################
  Scenario 2 — Retry Success (第一次 503 -> Backoff -> Retry 成功)
##################################################

🚀 [1. 准备阶段] 初始化任务
   -> 幂等键: 08f8be72-9825-432d-a6ca-336716c1020b
   -> 全局 Deadline 预算: 5.00s

[API] Attempt #1
[API] 503

[Policy]
retry_count = 0/3
remaining_budget = 5.00s
cancelled = False

[Retry]
backoff = 0.27s

[Retry] 执行休眠 0.27s...

[API] Attempt #2
[API] 200 OK

✅ [Final State] SUCCESS - 操作执行成功

[Execution Result]: {'status': 'SUCCESS', 'detail': {'success': True, 'message': 'HR Bonus applied.'}}


################

In [6]:
import time
from enum import Enum
from typing import Optional


class CircuitState(Enum):
    CLOSED = "CLOSED"
    OPEN = "OPEN"
    HALF_OPEN = "HALF_OPEN"


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 30.0):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown

        # 服务健康状态（跨所有请求共享）
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at: Optional[float] = None

        # 标记是否已有 probe 在执行（预留给 Half-Open 并发控制）
        self.half_open_probe_in_flight = False

    def can_call(self) -> bool:
        """根据当前状态与 Cooldown 决定是否放行请求"""
        now = time.time()

        if self.state == CircuitState.CLOSED:
            return True

        if self.state == CircuitState.OPEN:
            # 严格防错校验：OPEN 状态下 opened_at 绝不能为 None，防御异常数据
            if self.opened_at is None:
                raise RuntimeError(
                    "[CircuitBreaker Error] Invalid state: Circuit is OPEN but 'opened_at' is None."
                )

            elapsed = now - self.opened_at

            # Cooldown 未结束 -> Fast Fail
            if elapsed < self.cooldown:
                return False

            # Cooldown 结束 -> 状态转换为 HALF_OPEN，放行探针请求
            print(
                f"\n[CircuitBreaker] ⏱️ Cooldown 结束 ({elapsed:.2f}s >= {self.cooldown}s)，状态转换: OPEN -> HALF_OPEN"
            )
            self.state = CircuitState.HALF_OPEN
            self.half_open_probe_in_flight = True
            return True

        if self.state == CircuitState.HALF_OPEN:
            # 如果当前已经有一个试探请求在执行中，其余并发请求直接 Fast Fail
            if self.half_open_probe_in_flight:
                print(
                    "[CircuitBreaker] 🛡️ HALF_OPEN 状态已有试探请求在执行中，拦截并发请求 (Fast Fail)"
                )
                return False
            
            self.half_open_probe_in_flight = True
            return True

        return False

    def record_success(self):
        """请求成功时的状态回调"""
        if self.state == CircuitState.HALF_OPEN:
            print(
                "[CircuitBreaker] 🟢 HALF_OPEN 试探请求成功！服务已恢复，状态转换: HALF_OPEN -> CLOSED"
            )
        else:
            print("[CircuitBreaker] 🟢 请求成功，重置失败计数")

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at = None
        self.half_open_probe_in_flight = False

    def record_failure(self):
        """请求失败时的状态回调"""
        now = time.time()

        if self.state == CircuitState.HALF_OPEN:
            # HALF_OPEN 试探失败，立即重新进入 OPEN 并更新 cooldown 计时起点
            print(
                "[CircuitBreaker] 🚨 HALF_OPEN 试探失败！重新触发熔断，状态转换: HALF_OPEN -> OPEN"
            )
            self.state = CircuitState.OPEN
            self.opened_at = now
            self.half_open_probe_in_flight = False
            return

        # CLOSED 状态下的失败累加
        self.failure_count += 1
        print(
            f"[CircuitBreaker] ⚠️ 连续失败计次: {self.failure_count}/{self.failure_threshold}"
        )

        if self.failure_count >= self.failure_threshold:
            print(
                f"[CircuitBreaker] 🚨 达到连续失败阈值 ({self.failure_threshold})！状态转换: CLOSED -> OPEN"
            )
            self.state = CircuitState.OPEN
            self.opened_at = now
            self.half_open_probe_in_flight = False

if __name__ == "__main__":
    # 创建一个全局共享的 Circuit Breaker 实例（连续失败 2 次熔断，Cooldown 2 秒）
    hr_api_breaker = CircuitBreaker(failure_threshold=2, cooldown=2.0)

    print("--- 1. 正常状态 (CLOSED) ---")
    print(f"can_call? {hr_api_breaker.can_call()}")  # True
    hr_api_breaker.record_failure()
    print(f"can_call? {hr_api_breaker.can_call()}")  # True (1/2)

    print("\n--- 2. 触发熔断 (CLOSED -> OPEN) ---")
    hr_api_breaker.record_failure()                  # 触发阈值 (2/2) -> OPEN
    print(f"Can call immediately? {hr_api_breaker.can_call()}") # False (Fast Fail)

    print("\n--- 3. Cooldown 未结束时 (OPEN Fast Fail) ---")
    time.sleep(1.0)
    print(f"Can call at 1.0s? {hr_api_breaker.can_call()}")     # False

    print("\n--- 4. Cooldown 结束后转换 (OPEN -> HALF_OPEN) ---")
    time.sleep(1.1)                                             # 累计等待 2.1s > 2.0s
    print(f"Can call probe at 2.1s? {hr_api_breaker.can_call()}") # True (转换为 HALF_OPEN)

    print("\n--- 5. HALF_OPEN 并发探针拦截测试 ---")
    # 模拟并发请求在第一个 probe 尚未返回时到达
    print(f"Concurrent call during probe? {hr_api_breaker.can_call()}") # False (拦截)

    print("\n--- 6. 试探失败重置 Cooldown (HALF_OPEN -> OPEN) ---")
    hr_api_breaker.record_failure()                            # 试探失败 -> 重置为 OPEN
    print(f"Can call right after failed probe? {hr_api_breaker.can_call()}") # False

    print("\n--- 7. Cooldown 再次结束后成功恢复 (OPEN -> HALF_OPEN -> CLOSED) ---")
    time.sleep(2.1)
    print(f"Can call probe again? {hr_api_breaker.can_call()}")  # True
    hr_api_breaker.record_success()                             # 试探成功 -> CLOSED
    print(f"Current State: {hr_api_breaker.state.value}")       # CLOSED

--- 1. 正常状态 (CLOSED) ---
can_call? True
[CircuitBreaker] ⚠️ 连续失败计次: 1/2
can_call? True

--- 2. 触发熔断 (CLOSED -> OPEN) ---
[CircuitBreaker] ⚠️ 连续失败计次: 2/2
[CircuitBreaker] 🚨 达到连续失败阈值 (2)！状态转换: CLOSED -> OPEN
Can call immediately? False

--- 3. Cooldown 未结束时 (OPEN Fast Fail) ---
Can call at 1.0s? False

--- 4. Cooldown 结束后转换 (OPEN -> HALF_OPEN) ---

[CircuitBreaker] ⏱️ Cooldown 结束 (2.10s >= 2.0s)，状态转换: OPEN -> HALF_OPEN
Can call probe at 2.1s? True

--- 5. HALF_OPEN 并发探针拦截测试 ---
[CircuitBreaker] 🛡️ HALF_OPEN 状态已有试探请求在执行中，拦截并发请求 (Fast Fail)
Concurrent call during probe? False

--- 6. 试探失败重置 Cooldown (HALF_OPEN -> OPEN) ---
[CircuitBreaker] 🚨 HALF_OPEN 试探失败！重新触发熔断，状态转换: HALF_OPEN -> OPEN
Can call right after failed probe? False

--- 7. Cooldown 再次结束后成功恢复 (OPEN -> HALF_OPEN -> CLOSED) ---

[CircuitBreaker] ⏱️ Cooldown 结束 (2.10s >= 2.0s)，状态转换: OPEN -> HALF_OPEN
Can call probe again? True
[CircuitBreaker] 🟢 HALF_OPEN 试探请求成功！服务已恢复，状态转换: HALF_OPEN -> CLOSED
Current State: CLOSED
